In [64]:
# Datasets must be downloaded from Kaggle (or manually collected) and transformed to meet expected folder and file structure
ARDUINO_DIR = "Arduino"
RAVDESS_DIR = "RAVDESS"
EMOTIONS_DIR = "Emotions"
SCREAMING_DIR = "Screaming"

AGGRESSIVE = "aggressive"
NEUTRAL = "neutral"

LABEL_TO_ID = {
    NEUTRAL: 0,
    AGGRESSIVE: 1
}

In [65]:
class InitialDataset():
    def __init__(self):
        self.records = []
    def add_record(self, source, path, label):
        self.records.append({"source": source, "path": path, "label": label})

In [66]:
import os
import glob

def process_arduino_dataset(initial_dataset):
    # Aggressive sub-folder
    files = glob.glob(os.path.join(ARDUINO_DIR, "Aggressive", "*.wav"))
    for file in files:
        initial_dataset.add_record("ARDUINO_DATASET", file, AGGRESSIVE)
    # Neutral sub-folder
    files = glob.glob(os.path.join(ARDUINO_DIR, "Neutral", "*.wav"))
    for file in files:
        initial_dataset.add_record("ARDUINO_DATASET", file, NEUTRAL)

def process_ravdess_dataset(initial_dataset):
    files = glob.glob(os.path.join(RAVDESS_DIR, "*.wav"))
    for file in files:
        filename = os.path.basename(file)
        parts = filename.replace(".wav", "").split("-")
        if parts[2] == "05" or (parts[2] == "06" and parts[3] == "02"): # emotion=angry or emotion=fearful and intensity=strong
            label = AGGRESSIVE
        else:
            label = NEUTRAL
        initial_dataset.add_record("RAVDESS_DATASET", file, label)

def process_emotions_dataset(initial_dataset):
    files = glob.glob(os.path.join(EMOTIONS_DIR, "*.wav"))
    for file in files:
        initial_dataset.add_record("EMOTIONS_DATASET", file, NEUTRAL)

def process_screaming_dataset(initial_dataset):
    # Screaming sub-folder
    files = glob.glob(os.path.join(SCREAMING_DIR, "Screaming", "*.wav"))
    for file in files:
        initial_dataset.add_record("SCREAMING_DATASET", file, AGGRESSIVE)
    # NotScreaming sub-folder
    files = glob.glob(os.path.join(SCREAMING_DIR, "NotScreaming", "*.wav"))
    for file in files:
        initial_dataset.add_record("SCREAMING_DATASET", file, NEUTRAL)

In [67]:
import pandas as pd

initial_dataset = InitialDataset()
process_arduino_dataset(initial_dataset)
process_ravdess_dataset(initial_dataset)
process_emotions_dataset(initial_dataset)
process_screaming_dataset(initial_dataset)

if len(initial_dataset.records) == 0:
    raise ValueError("No audio files found")

records_df = pd.DataFrame(initial_dataset.records)

In [68]:
def print_dataset_diagnostics(df):
    print("================ DATASET DIAGNOSTICS ================")
    print("\nTotal samples:")
    print(len(df))
    print("\nSource distribution:")
    print(df["source"].value_counts())
    print("\nLabel distribution:")
    print(df["label"].value_counts())
    print("\n=====================================================")

print_dataset_diagnostics(records_df)

================ DATASET DIAGNOSTICS ================

Total samples:
5014

Source distribution:
source
SCREAMING_DATASET    3493
RAVDESS_DATASET      1440
EMOTIONS_DATASET       68
ARDUINO_DATASET        13
Name: count, dtype: int64

Label distribution:
label
neutral       3858
aggressive    1156
Name: count, dtype: int64



In [69]:
from sklearn.model_selection import train_test_split

SEED = 42

records_df = records_df.sample(frac=1, random_state=SEED).reset_index(drop=True)

train_df, temp_df = train_test_split(
    records_df,
    test_size=0.3, # Needed to increase for stratify to have enough samples in subsequent val/test split
    random_state=SEED,
    stratify=records_df[["source", "label"]]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=SEED,
    stratify=temp_df[["source", "label"]]
)

print_dataset_diagnostics(train_df)
print_dataset_diagnostics(val_df)
print_dataset_diagnostics(test_df)

================ DATASET DIAGNOSTICS ================

Total samples:
3509

Source distribution:
source
SCREAMING_DATASET    2444
RAVDESS_DATASET      1008
EMOTIONS_DATASET       48
ARDUINO_DATASET         9
Name: count, dtype: int64

Label distribution:
label
neutral       2700
aggressive     809
Name: count, dtype: int64

================ DATASET DIAGNOSTICS ================

Total samples:
752

Source distribution:
source
SCREAMING_DATASET    524
RAVDESS_DATASET      216
EMOTIONS_DATASET      10
ARDUINO_DATASET        2
Name: count, dtype: int64

Label distribution:
label
neutral       579
aggressive    173
Name: count, dtype: int64

================ DATASET DIAGNOSTICS ================

Total samples:
753

Source distribution:
source
SCREAMING_DATASET    525
RAVDESS_DATASET      216
EMOTIONS_DATASET      10
ARDUINO_DATASET        2
Name: count, dtype: int64

Label distribution:
label
neutral       579
aggressive    174
Name: count, dtype: int64



In [70]:
import librosa
import numpy as np

SAMPLE_RATE = 16000
DURATION = 6
N_FFT = 2048
HOP_LENGTH = 512
N_MELS = 128

def load_audio(filepath):
    audio, sr = librosa.load(filepath, sr=SAMPLE_RATE)
    
    current_length = len(audio)
    target_length = SAMPLE_RATE * DURATION
    if current_length > target_length:
        start = (current_length - target_length) // 2
        end = start + target_length
        audio = audio[start:end]
    elif current_length < target_length:
        total_padding = target_length - current_length
        pad_left = total_padding // 2
        pad_right = total_padding - pad_left
        audio = np.pad(audio, (pad_left, pad_right))

    mel = librosa.feature.melspectrogram(
        y=audio,
        sr=sr,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS
    )

    mel_db = librosa.power_to_db(mel, ref=np.max)
    mel_db = (mel_db - mel_db.mean()) / (mel_db.std() + 1e-6)
    return mel_db.astype(np.float32)

In [71]:
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

BATCH_SIZE = 32

SOURCE_LOSS_WEIGHTS = {
    "ARDUINO_DATASET": 1.5,
    "SCREAMING_DATASET": 0.5,
    "RAVDESS_DATASET": 1.0,
    "EMOTIONS_DATASET": 1.0
}

SOURCE_WEIGHTS = {
    "ARDUINO_DATASET": 1.5,
    "SCREAMING_DATASET": 0.5,
    "RAVDESS_DATASET": 1.0,
    "EMOTIONS_DATASET": 1.0
}

class AudioDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
    def __len__(self):
        return len(self.df)
    def __getitem__(self, ind):
        row = self.df.iloc[ind]
        filepath = row["path"]
        mel = load_audio(filepath)
        mel = torch.tensor(mel).unsqueeze(0) # CNN expects channels-first
        
        label = LABEL_TO_ID[row["label"]]
        source_weight = SOURCE_LOSS_WEIGHTS[row["source"]]
        
        return (mel, torch.tensor(label, dtype=torch.long), torch.tensor(source_weight, dtype=torch.float32))

sample_weights = []
for _, row in train_df.iterrows():
    weight = SOURCE_WEIGHTS[row["source"]]
    sample_weights.append(weight)
sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

train_dataset = AudioDataset(train_df)
val_dataset = AudioDataset(val_df)
test_dataset = AudioDataset(test_df)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset,  batch_size=BATCH_SIZE)

In [72]:
import torch
import torch.nn as nn
from tqdm.auto import tqdm
from sklearn.metrics import classification_report

EPOCHS = 20
LEARNING_RATE = 1e-3

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

class AudioCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),
            # 
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            # 
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            # 
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 4))
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 2)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = AudioCNN().to(DEVICE)

class_weights = torch.tensor([1.0, 1.5], dtype=torch.float32).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights, reduction="none")
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

for epoch in tqdm(range(EPOCHS), desc="Training"):
    model.train()

    train_loss = 0
    train_correct = 0
    train_total = 0

    train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1} [Train]", leave=False)

    for inputs, targets, source_weights in train_bar:
        inputs = inputs.to(DEVICE)
        targets = targets.to(DEVICE)
        source_weights = source_weights.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(inputs)

        # loss = criterion(outputs, targets)
        losses = criterion(outputs, targets)
        weighted_losses = losses * source_weights
        loss = weighted_losses.mean()

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

        predictions = outputs.argmax(dim=1)

        train_correct += (predictions == targets).sum().item()
        train_total += targets.size(0)

        train_bar.set_postfix({
            "loss": f"{loss.item():.4f}",
            "acc": f"{train_correct / max(train_total, 1):.4f}"
        })
        

    train_acc = train_correct / train_total

    model.eval()

    val_loss = 0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        val_bar = tqdm(val_loader, desc=f"Epoch {epoch+1} [Val]", leave=False)

        for inputs, targets, source_weights  in val_bar:
            inputs = inputs.to(DEVICE)
            targets = targets.to(DEVICE)
            source_weights = source_weights.to(DEVICE)

            outputs = model(inputs)

            # loss = criterion(outputs, targets)
            losses = criterion(outputs, targets)
            weighted_losses = losses * source_weights
            loss = weighted_losses.mean()

            val_loss += loss.item()

            predictions = outputs.argmax(dim=1)

            val_correct += (predictions == targets).sum().item()
            val_total += targets.size(0)

            val_bar.set_postfix({
                "loss": f"{loss.item():.4f}",
                "acc": f"{val_correct / max(val_total, 1):.4f}"
            })

    val_acc = val_correct / val_total

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f}"
    )

Training:   0%|          | 0/20 [00:00<?, ?it/s]

Epoch 1 [Train]:   0%|          | 0/110 [00:00<?, ?it/s]

Epoch 1 [Val]:   0%|          | 0/24 [00:00<?, ?it/s]

Epoch 1/20 | Train Loss: 43.1029 | Train Acc: 0.8068 | Val Loss: 8.6094 | Val Acc: 0.7992


Epoch 2 [Train]:   0%|          | 0/110 [00:00<?, ?it/s]

...

Epoch 19 [Val]:   0%|          | 0/24 [00:00<?, ?it/s]

Epoch 19/20 | Train Loss: 12.7630 | Train Acc: 0.9276 | Val Loss: 8.4668 | Val Acc: 0.8590


Epoch 20 [Train]:   0%|          | 0/110 [00:00<?, ?it/s]

Epoch 20 [Val]:   0%|          | 0/24 [00:00<?, ?it/s]

Epoch 20/20 | Train Loss: 11.4369 | Train Acc: 0.9367 | Val Loss: 10.6480 | Val Acc: 0.8351


In [76]:
from datetime import datetime

model.eval()

all_preds = []
all_targets = []

with torch.no_grad():
    for inputs, targets, _ in test_loader:
        inputs = inputs.to(DEVICE)
        outputs = model(inputs)
        predictions = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(predictions)
        all_targets.extend(targets.numpy())


print(classification_report(
    all_targets,
    all_preds,
    target_names=[NEUTRAL, AGGRESSIVE]
))

os.makedirs("models", exist_ok=True)
MODEL_PATH = f"models/aggression_cnn_{datetime.now().strftime("%Y-%m-%d_%H-%M-%S")}.pth"
torch.save(model.state_dict(), MODEL_PATH)
print(f"Saved model to: {MODEL_PATH}")

              precision    recall  f1-score   support

     neutral       0.94      0.81      0.87       579
  aggressive       0.56      0.82      0.67       174

    accuracy                           0.81       753
   macro avg       0.75      0.81      0.77       753
weighted avg       0.85      0.81      0.82       753

Saved model to: models/aggression_cnn_2026-05-19_16-11-56.pth
